### DAS Processing

In [ ]:
# Import necessary dependencies
import gc
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import detrend
from scipy.signal import welch

sys.path.append('..')

from src.utils import nextpow2
from src.ani import bandpass_filter_tukey, temporal_normalization, spectral_whitening
from src.plots import plot_das_wavefield, plot_das_psd, plot_das_psd_2d

In [ ]:
# Define paths
das_path = os.path.join('..', 'data', 'raw_urban', '20210901', '20210901_000000.npz')

# Load das data
das_data = np.load(das_path)

# Inspect variables inside 
for key in das_data:
    print(f'{key}')

# DAS interrogator parameters
dt = 0.004 # Sampling rate in sec (or 250 Hz)
gauge_len = 16.0
dx = 8.16

In [ ]:
# Extract das data (channels, time)
das_array = das_data['data']
dt = das_data['dt']                 # Sampling rate in sec
fs = 1.0 / dt
N = das_data['data'].shape[1]       # Number of samples
T = N * dt                      # Total duration

print('DAS array shape:', das_array.shape)
print('Time axis shape:', das_data['t_axis'].shape)
print('Channel positions:', das_data['x_axis'].shape)
print(f'Sampling rates: {dt} s')
print(f'Total duration: {T/60:.2f} minutes')

In [ ]:
# Set preprocessing parameters
fs              = 250        # sampling frequency (Hz)
f1, f2          = 1, 10    # bandpass filter corners
Decimation      = 1          # if not 1, decimation factor after filtering
diff = False                 # whether to differentiate (strain → strain rate)
min_length = 60              # length of the segment in preprocessing, in sec, if shorter than this length, skip the file

#### Detrend

In [ ]:
x_detrended = detrend(das_array, axis=-1)

plot_das_wavefield(
    x_detrended,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="Detrended DAS", 
    figsize=(10, 6)
)

#### Bandpass Filter

In [ ]:
x_filtered = bandpass_filter_tukey(x_detrended, fs=fs, f1=f1, f2=f2, alpha=0.05, order=4)

plot_das_wavefield(
    x_filtered,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="Filtered DAS (1-10 Hz)", 
    figsize=(10, 6)
)

#### Remove Median

In [ ]:
# Calculate spatial median (axis=0) and subtract it from all channels
x_med = x_filtered - np.median(x_filtered, axis=0)

plot_das_wavefield(
    x_med,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="Median-Removed DAS", 
    figsize=(10, 6)
)

#### Temporal Normalization

In [ ]:
# Set RAM window length (seconds)
# If 0.0, the function applies 1-bit normalization.
onebit = 0 
ram_win = 1

# Apply temporal normalization
x_tem_norm_onebit = temporal_normalization(x_med, fs=fs, window_time=onebit)
x_tem_norm_ram = temporal_normalization(x_med, fs=fs, window_time=ram_win)

plot_das_wavefield(
    x_tem_norm_onebit,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="1-bit Normalized DAS", 
    figsize=(10, 6)
)

plot_das_psd(
    data=x_tem_norm_onebit,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,           
    flim=None,       
    psd_ylim=None,              
    xscale="log",        
    ylabel="PSD (dB)",
    figsize=(10, 6), 
    title="1-bit Normalized DAS"
)

plot_das_psd_2d(
    data=x_tem_norm_onebit,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,     
    flim=(0, 25),                            
    figsize=(10, 6), 
    title="1-bit Normalized DAS"
)

#######################

plot_das_wavefield(
    x_tem_norm_ram,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="RAM Normalized DAS", 
    figsize=(10, 6)
)

plot_das_psd(
    data=x_tem_norm_ram,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,           
    flim=None,       
    psd_ylim=None,              
    xscale="log",        
    ylabel="PSD (dB)",
    figsize=(10, 6), 
    title="RAM Normalized DAS"
)

plot_das_psd_2d(
    data=x_tem_norm_ram,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,     
    flim=(0, 25),                            
    figsize=(10, 6), 
    title="RAM Normalized DAS"
)

#### Spectral Whitening

**1-Bit + Phase-only**

In [ ]:
# Set Whitening Parameters
device = torch.device("cpu")

is_spectral_whitening = True
window_freq_phase = 0.0   # 0.0 = phase-only whitening; > 0 = running-mean amplitude whitening
window_freq_ram = 1.0

X = torch.as_tensor(x_tem_norm_onebit, dtype=torch.float32, device=device)
nch, nt = X.shape

if is_spectral_whitening:
    # Choose FFT length (power-of-two is optimal for speed)
    nfft = nextpow2(nt)
    df = fs / nfft  # Hz per frequency bin

    # Real FFT along the time axis
    R = torch.fft.rfft(X, n=nfft, dim=-1)

    # Apply spectral whitening (from our custom function)
    Rw = spectral_whitening(
        rfftdata=R,
        df=float(df),
        window_freq=float(window_freq_phase),
        f1=float(f1),
        f2=float(f2),
    )

    # Inverse FFT back to time domain, and crop padded values
    Xw = torch.fft.irfft(Rw, n=nfft, dim=-1)[..., :nt]
else:
    Xw = X

x_whiten = Xw.detach().cpu().numpy().astype(np.float32, copy=False)

plot_das_wavefield(
    x_whiten,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="One-bit Normalized + Phase-only Whitened DAS", 
    figsize=(10, 6)
)

plot_das_psd(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,           
    flim=None,       
    psd_ylim=None,              
    xscale="log",        
    ylabel="PSD (dB)",
    figsize=(10, 6), 
    title="One-bit Normalized + Phase-only Whitened DAS"
)

plot_das_psd_2d(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,     
    flim=(0, 25),                            
    figsize=(10, 6), 
    title="One-bit Normalized + Phase-only Whitened DAS"
)

del X, Xw, x_whiten
gc.collect()

**RAM + Phase-only**

In [ ]:
# Set Whitening Parameters
device = torch.device("cpu")

is_spectral_whitening = True
window_freq_phase = 0.0   # 0.0 = phase-only whitening; > 0 = running-mean amplitude whitening
window_freq_ram = 1.0

X = torch.as_tensor(x_tem_norm_ram, dtype=torch.float32, device=device)
nch, nt = X.shape

if is_spectral_whitening:
    # Choose FFT length (power-of-two is optimal for speed)
    nfft = nextpow2(nt)
    df = fs / nfft  # Hz per frequency bin

    # Real FFT along the time axis
    R = torch.fft.rfft(X, n=nfft, dim=-1)

    # Apply spectral whitening (from our custom function)
    Rw = spectral_whitening(
        rfftdata=R,
        df=float(df),
        window_freq=float(window_freq_phase),
        f1=float(f1),
        f2=float(f2),
    )

    # Inverse FFT back to time domain, and crop padded values
    Xw = torch.fft.irfft(Rw, n=nfft, dim=-1)[..., :nt]
else:
    Xw = X

x_whiten = Xw.detach().cpu().numpy().astype(np.float32, copy=False)

plot_das_wavefield(
    x_whiten,
    fs=fs,
    dx=dx,
    start_sec=0.0,
    duration_sec=60.0,     
    start_chan=None, 
    end_chan=None, 
    pclip=99,            
    title="RAM Normalized + Phase-only Whitened DAS", 
    figsize=(10, 6)
)

plot_das_psd(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,           
    flim=None,       
    psd_ylim=None,              
    xscale="log",        
    ylabel="PSD (dB)",
    figsize=(10, 6), 
    title="RAM Normalized + Phase-only Whitened DAS"
)

plot_das_psd_2d(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,     
    flim=(0, 25),                            
    figsize=(10, 6), 
    title="RAM Normalized + Phase-only Whitened DAS"
)

del X, Xw, x_whiten
gc.collect()

**1-Bit + RAM**

In [ ]:
# Set Whitening Parameters
device = torch.device("cpu")
is_spectral_whitening = True
window_freq_ram = 1.0
chunk_size = 10 

X = torch.as_tensor(x_tem_norm_onebit, dtype=torch.float32, device=device)
nch, nt = X.shape
Xw = torch.empty_like(X)

if is_spectral_whitening:
    nfft = nextpow2(nt)
    df = fs / nfft  

    # 1. FORCE PYTORCH TO SHUT OFF BACKPROPAGATION TRACKING
    with torch.inference_mode():
        for ch_start in range(0, nch, chunk_size):
            ch_end = min(ch_start + chunk_size, nch)
            X_chunk = X[ch_start:ch_end, :]
            
            R_chunk = torch.fft.rfft(X_chunk, n=nfft, dim=-1)
            Rw_chunk = spectral_whitening(
                rfftdata=R_chunk, df=float(df), window_freq=float(window_freq_ram),
                f1=float(f1), f2=float(f2),
            )
            
            Xw[ch_start:ch_end, :] = torch.fft.irfft(Rw_chunk, n=nfft, dim=-1)[..., :nt]
            
            # 2. MANUALLY DELETE TEMPORARY VARIABLES
            del X_chunk, R_chunk, Rw_chunk
            
            # 3. FORCE PYTHON TO EMPTY THE TRASH IMMEDIATELY
            gc.collect() 
else:
    Xw = X

x_whiten = Xw.detach().cpu().numpy().astype(np.float32, copy=False)

plot_das_wavefield(
    x_whiten, fs=fs, dx=dx, start_sec=0.0, duration_sec=60.0,     
    pclip=99, title="One-bit Normalized + RAM Whitened DAS", figsize=(10, 6)
)

plot_das_psd(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,           
    flim=None,       
    psd_ylim=None,              
    xscale="log",        
    ylabel="PSD (dB)",
    figsize=(10, 6), 
    title="One-bit Normalized + RAM Whitened DAS"
)

plot_das_psd_2d(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,     
    flim=(0, 25),                            
    figsize=(10, 6), 
    title="One-bit Normalized + RAM Whitened DAS"
)

del X, Xw, x_whiten
gc.collect()

**RAM + RAM**

In [ ]:
# Set Whitening Parameters
device = torch.device("cpu")
is_spectral_whitening = True
window_freq_ram = 1.0
chunk_size = 10 

X = torch.as_tensor(x_tem_norm_ram, dtype=torch.float32, device=device)
Xw = torch.empty_like(X)

if is_spectral_whitening:
    nfft = nextpow2(nt)
    df = fs / nfft  

    with torch.inference_mode():
        for ch_start in range(0, nch, chunk_size):
            ch_end = min(ch_start + chunk_size, nch)
            X_chunk = X[ch_start:ch_end, :]
            
            R_chunk = torch.fft.rfft(X_chunk, n=nfft, dim=-1)
            Rw_chunk = spectral_whitening(
                rfftdata=R_chunk, df=float(df), window_freq=float(window_freq_ram),
                f1=float(f1), f2=float(f2),
            )
            
            Xw[ch_start:ch_end, :] = torch.fft.irfft(Rw_chunk, n=nfft, dim=-1)[..., :nt]
            
            del X_chunk, R_chunk, Rw_chunk
            gc.collect()
else:
    Xw = X

x_whiten = Xw.detach().cpu().numpy().astype(np.float32, copy=False)

plot_das_wavefield(
    x_whiten, fs=fs, dx=dx, start_sec=0.0, duration_sec=60.0,     
    pclip=99, title="RAM Normalized + RAM Whitened DAS", figsize=(10, 6)
)

plot_das_psd(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,           
    flim=None,       
    psd_ylim=None,              
    xscale="log",        
    ylabel="PSD (dB)",
    figsize=(10, 6), 
    title="RAM Normalized + RAM Whitened DAS"
)

plot_das_psd_2d(
    data=x_whiten,
    fs=fs,
    dx=dx,
    start_chan=0,
    end_chan=None,
    nperseg=4096,     
    flim=(0, 25),                            
    figsize=(10, 6), 
    title="RAM Normalized + RAM Whitened DAS"
)

del X, Xw, x_whiten
gc.collect()